# Sessions J0–J4: the four J. Chem. Educ. examples

Prompts exactly as in the paper. Each prompt cell starts a fresh session with `new_chat()`; a multi-turn conversation continues in the cells below it without `new_chat()`. Where the paper's run answered a question from Dynalyst, the reply is in a cell after the prompt, as `reply('yes')`: it sends the reply only if Dynalyst has asked a question in this run, and otherwise does nothing.

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Run the **setup cell**, choose the model and paste its API key.
2. Run the **files cell**: it copies the extra models, the R01 time course and the papers from `model_files/` (next to this notebook) into `data/`.
3. Run any prompt cell. Prompts that use BioModels, PubMed or OmniPath need internet.
4. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup and files cells again.

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

In [ ]:
# Copy the extra model files, data and papers into the project (only files not already there)
import shutil
FILES = Path("model_files") if Path("model_files").exists() else ROOT / "notebooks" / "benchmarks" / "model_files"
DEST = {".xml": config.DATA_DIR / "sbml_models", ".ode": config.DATA_DIR / "sbml_models",
        ".csv": config.DATA_DIR / "timecourses", ".pdf": config.PAPERS_DIR}
for f in sorted(FILES.iterdir()):
    d = DEST.get(f.suffix.lower())
    if d and not (d / f.name).exists():
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
        print("added", d / f.name)
print("model files ready")

## Examples of Sriram (2026) and Harika and Sriram (2026) (main text: RQ1, Table 2, Figs 9 and 10; S1 File, Section S8)

Each session is a fresh conversation; its prompts are sent in order in the same conversation. P1.1 needs the EMBED paper (Harika and Sriram, ACS Omega 2026) as a PDF in data/papers/; it is not shipped here.

### Session J0: Feedback loops of the four J. Chem. Educ. examples: Jacobian vs CRNT

In [ ]:
new_chat()
ask('The models jce1_production_loss.xml, jce2_mm_autoregulation.xml, jce3_hill_autoregulation.xml and jce4_mutual_inhibition.xml are the four examples of Sriram, J. Chem. Educ. 2026. For each model, use the Jacobian at its steady states to tell me whether it has feedback loops, of which kind (positive or negative; self-regulation of one species or a loop between species), and how many positive and how many negative feedback loops it has.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('Now analyse the same four models with chemical reaction network theory: for each, its reaction network, deficiency, and whether CRNT allows more than one steady state for some rate constants. Then explain how this differs from the Jacobian answer: which of the two needed the values of the kinetic constants and which did not?')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('For each of the four models, for which kinetic parameter can it be bistable, and over what range of that parameter? Check each claim with a bifurcation diagram.')

### Session J1: Example 1: production and loss

In [ ]:
new_chat()
ask('Find all steady states of jce1_production_loss.xml and their stability.')

In [ ]:
ask('Simulate it from P = 0, 0.5, 2 and 4 on one plot, and plot dP/dt against time for the run from P = 0.')

In [ ]:
ask('Can this model have a bifurcation in any of its parameters? Answer from its structure and its dimension.')

### Session J2: Example 2: Michaelis-Menten autoregulation

In [ ]:
new_chat()
ask('Draw the bifurcation diagram of jce2_mm_autoregulation.xml in k1 from 0 to 3 and tell me which bifurcation it has, where, and what happens to the steady states on either side.')

In [ ]:
ask('Show time courses at k1 = 0.5 and k1 = 2 from P = 0.01 and P = 2 on one figure per value.')

In [ ]:
ask('Give me the Jacobian at each steady state for k1 = 0.5 and k1 = 2 and say how its sign explains the stability.')

### Session J3: Example 3: Hill autoregulation

In [ ]:
new_chat()
ask('Draw the bifurcation diagram of jce3_hill_autoregulation.xml in k1 from 0 to 4. Which bifurcations does it have, and is the model bistable anywhere?')

In [ ]:
ask('At k1 = 3, find all steady states with their stability and show trajectories from P = 0.2, 0.5 and 3.')

In [ ]:
ask('Follow the fold points in the two parameters k1 and n, with n from 1 to 4. Below which value of n does the bistability disappear?')

In [ ]:
reply('yes')   # Dynalyst asked (parameter); reply yes

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('Run a Gillespie simulation at k1 = 3 with a system size of 50 molecules per unit concentration, starting from the high state, and say whether the noise makes it switch to the low state within 200 time units.')

### Session J4: Example 4: mutual inhibition

In [ ]:
new_chat()
ask('Draw the bifurcation diagram of jce4_mutual_inhibition.xml in k1 from 0 to 2 and identify the bifurcation. Plot both P1 and P2.')

In [ ]:
ask('Plot the phase plane of jce4_mutual_inhibition.xml at k1 = 1.5 with nullclines and a few trajectories.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

In [ ]:
ask('Draw the interaction network from the Jacobian at each steady state for k1 = 1.5 and list its feedback loops with their signs.')

In [ ]:
ask('Set n = 1 and draw the bifurcation diagram in k1 from 0 to 5 again. Does the bifurcation still occur? Explain why or why not.')

In [ ]:
reply('yes')   # Dynalyst asked (widen); reply yes

In [ ]:
ask('How many independent variables does this model have, and which bifurcations does that dimension allow?')